# Milestone 2 — Entity Extraction & Graph Building
### AI-Based Knowledge Graph Builder for Enterprise Intelligence

---

## Objective
Extract entities and relationships from enterprise support ticket data using a Large Language Model (LLM), construct structured knowledge graph triples, and store the knowledge graph in Neo4j for querying and visualization.

## Tasks Covered
- Apply LLM-based Named Entity Recognition (NER) using Mistral
- Extract structured triples (Entity → Relation → Entity) from ticket descriptions
- Create rule-based structured triples from ticket columns
- Merge all triples into a final knowledge graph dataset
- Build an in-memory graph using NetworkX
- Store results in Neo4j graph database
- Validate the graph using Cypher queries



---
## Step-1 Environment Setup and Library Initialization

### Objective
Install and import all required Python libraries needed for this milestone.

### Libraries Used
| Library | Purpose |
|---|---|
| `pandas` | Load and manipulate the cleaned ticket dataset |
| `ollama` | Interface with locally running Mistral LLM for NER |
| `re` | Regular expressions to parse LLM output into triples |
| `networkx` | Build and analyze the in-memory knowledge graph |
| `neo4j` | Connect to Neo4j graph database and store triples |
| `transformers` | Access HuggingFace pretrained transformer models |
| `torch` | PyTorch deep learning framework for model inference |

In [ ]:
# Install required libraries
!pip install pandas transformers torch neo4j networkx openpyxl ollama

In [ ]:

import pandas as pd        # Data manipulation
import numpy as np         # Numerical operations
import re                  # Regular expressions for parsing LLM output
import json                # JSON handling
import os                  # File path operations
import networkx as nx      # Knowledge graph construction
import ollama              # Mistral LLM interface via Ollama

from transformers import AutoTokenizer, AutoModelForCausalLM 
from neo4j import GraphDatabase                               

print('✅ All libraries imported successfully!')

---
## Step 2 — Dataset Loading and Initial Inspection

### Objective
Load the preprocessed enterprise support ticket dataset produced in Milestone 1.

### Why This Step?
The `cleaned_tickets.xlsx` file is the output of Milestone 1 (Data Ingestion). It contains 8,469 cleaned and enriched support tickets with 21 columns. The most important columns for this milestone are:
- `Ticket Description` — Unstructured text used for LLM-based NER
- `Product Purchased` — Entity used as the subject in LLM triples
- `Ticket Subject` — Entity representing the type of issue
- `Ticket Priority`, `Ticket Channel`, `Ticket Status` — Used for structured triples

In [ ]:
# Load the cleaned dataset 
df = pd.read_excel('cleaned_tickets.xlsx')

# Display basic dataset information
print('=' * 50)
print('DATASET OVERVIEW')
print('=' * 50)
print(f'Total Tickets  : {len(df):,}')
print(f'Total Columns  : {df.shape[1]}')
print(f'Memory Usage   : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()
print('Columns Available:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

print()
df.head(3)

---
## Step 3 — Test Mistral LLM Connection via Ollama

### Objective
Verify that the Mistral Large Language Model is running locally via Ollama before processing the full dataset.

### What is Ollama?
Ollama is a tool that allows running open-source Large Language Models locally on your machine. It eliminates the need for cloud APIs or internet connectivity. We use it to run **Mistral 7B** — a powerful open-source LLM developed by Mistral AI.

### Why Mistral?
Mistral 7B is chosen because:
- It runs efficiently on local hardware without a GPU
- It produces high-quality structured outputs for NER tasks
- It is free and does not require API keys
- It follows prompt instructions accurately

In [ ]:
# Test Mistral LLM connection via Ollama
# This sends a simple message to verify the model is running

response = ollama.chat(
    model='mistral',                          # Mistral 7B model
    messages=[
        {'role': 'user',                      # We are the user
         'content': 'Say hello in one sentence.'}  # Simple test prompt
    ]
)


print('✅ Mistral LLM Connection Successful!')
print('   Model Response:', response['message']['content'])

---
## Step 4 — Test Triple Extraction on Sample Text

### Objective
Test the triple extraction pipeline on a small sample before running on the full dataset.

### What is a Triple?
A triple is the fundamental unit of a knowledge graph:
```
(Subject, Predicate, Object)
(Entity,  Relation,  Entity)

Example:
(LG Smart TV, has_issue, Overheating)
(Overheating, resolved_by, Factory Reset)
```
- **Subject** = the entity performing or experiencing something
- **Predicate** = the relationship between subject and object
- **Object** = the entity being affected or described



In [ ]:
# Sample ticket text for testing the extraction pipeline
sample_text = """
Customer reported battery overheating issue in LG Smart TV.
Suggested resolution is factory reset.
"""

# Prompt instructs the LLM to extract triples in a specific format
# The format (Entity, Relation, Entity) makes parsing with regex easy
prompt = f"""
You are an AI system that extracts knowledge graph triples.

Extract triples strictly in this format:
(Entity, Relation, Entity)

Text:
{sample_text}

Only output triples. No explanations.
"""

response = ollama.chat(
    model='mistral',
    messages=[{'role': 'user', 'content': prompt}]
)

print('✅ Sample Triple Extraction Output:')
print('-' * 40)
print(response['message']['content'])

---
## Step 5 — Parse LLM Output Using Regular Expressions

### Objective
Convert the LLM's raw text output into a structured Python list of triples.

### Why Regex?
The LLM returns text like:
```
1. (Customer, reports, Battery_Overheating_Issue)
2. (LG_Smart_TV, has, Battery_Overheating_Issue)
```
We cannot use this text directly. We use `re.findall()` to extract only the content inside brackets `( )` and then split by comma to get the 3 parts of each triple.



In [ ]:
# Parse raw LLM output into structured triples using regex
raw_output = response['message']['content']


triples = re.findall(r'\((.*?)\)', raw_output)

parsed_triples = []

for triple in triples:
    parts = [part.strip() for part in triple.split(',')]


    if len(parts) == 3:
        parsed_triples.append(parts)

print('✅ Parsed Triples:')
print('-' * 50)
for t in parsed_triples:
    print(f'  Subject  : {t[0]}')
    print(f'  Predicate: {t[1]}')
    print(f'  Object   : {t[2]}')
    print()

print(f'Total triples extracted: {len(parsed_triples)}')

---
## Step 6 — Structured Triple Extraction (Rules-Based Engine)

### Objective
Extract structured triples directly from the ticket's structured columns without using the LLM.

### What are Structured Triples?
Structured triples come from clearly defined column relationships in the dataset. These are rule-based — no AI needed:

```
Column-based Triple Examples:
(LG Smart TV,    HAS_ISSUE,     Product Setup)
(Ticket_1,       HAS_PRIORITY,  Critical)
(Ticket_1,       SUBMITTED_VIA, Email)
(Ticket_1,       HAS_STATUS,    Open)
(Ticket_1,       IS_TYPE,       Technical Issue)
(LG Smart TV,    RAISED,        Ticket_1)
```



In [ ]:
# Rules-Based Structured Triple Extraction
# Extracts triples directly from ticket columns

structured_triples = []

for _, row in df.iterrows():

    # Create a unique ticket identifier
    ticket_id = 'Ticket_' + str(row['Ticket ID'])

    # Extract key column values
    product  = str(row['Product Purchased'])
    subject  = str(row['Ticket Subject'])
    priority = str(row['Ticket Priority'])
    channel  = str(row['Ticket Channel'])
    status   = str(row['Ticket Status'])
    t_type   = str(row['Ticket Type'])

    # Triple 1: Product has a specific issue/subject
    # e.g. (LG Smart TV, HAS_ISSUE, Overheating)
    structured_triples.append([product,   'HAS_ISSUE',     subject])

    # Triple 2: Ticket has a priority level
    # e.g. (Ticket_1, HAS_PRIORITY, Critical)
    structured_triples.append([ticket_id, 'HAS_PRIORITY',  priority])

    # Triple 3: Ticket submitted via a channel
    # e.g. (Ticket_1, SUBMITTED_VIA, Email)
    structured_triples.append([ticket_id, 'SUBMITTED_VIA', channel])

    # Triple 4: Ticket has a current status
    # e.g. (Ticket_1, HAS_STATUS, Open)
    structured_triples.append([ticket_id, 'HAS_STATUS',    status])

    # Triple 5: Ticket belongs to a ticket type
    # e.g. (Ticket_1, IS_TYPE, Technical Issue)
    structured_triples.append([ticket_id, 'IS_TYPE',       t_type])

    # Triple 6: Product raised/created the ticket
    # e.g. (LG Smart TV, RAISED, Ticket_1)
    structured_triples.append([product,   'RAISED',        ticket_id])

# Convert list to DataFrame
structured_df = pd.DataFrame(
    structured_triples,
    columns=['Subject', 'Predicate', 'Object']
)

# Remove duplicate triples
structured_df = structured_df.drop_duplicates()

# Save to CSV for later use
structured_df.to_csv('structured_triples.csv', index=False)

print('✅ Structured Triple Extraction Complete!')
print(f'   Total Triples Extracted : {len(structured_df):,}')
print(f'   Unique Subjects         : {structured_df["Subject"].nunique()}')
print(f'   Unique Predicates       : {structured_df["Predicate"].nunique()}')
print(f'   Unique Objects          : {structured_df["Object"].nunique()}')
print()
structured_df.head(10)

---
## Step 7 — LLM-Based NER and Relation Extraction (Mistral)

### Objective
Use Mistral LLM to perform Named Entity Recognition (NER) and extract semantic relationships from unstructured ticket descriptions.





In [ ]:
# LLM-Based Named Entity Recognition and Relation Extraction
# Uses Mistral to extract semantic triples from ticket descriptions

llm_triples = []  # Store all extracted triples

# Process first 20 tickets (increase df.head(20) to process more)
for index, row in df.head(20).iterrows():

    # Extract ticket description and product name
    text    = row['Ticket Description']  # Unstructured text
    product = row['Product Purchased']   # Fixed subject entity

    # Structured prompt for consistent triple extraction
    # Subject is fixed as the product — only problem and action extracted
    prompt = f"""
The product involved is: {product}

From the text below, identify:
1. What specific problem the product is experiencing.
2. What action is required to fix it.

Return only valid triples in this exact format:
({product}, EXPERIENCING, Specific problem)
({product}, REQUIRED_ACTION, Specific action)

Rules:
- Do not create new entities or change the product name
- Do not use the word 'User' as a subject
- Do not use vague placeholders like 'unknown' or 'issue'
- If the text is unclear, return nothing

Text:
{text}
"""

    # Send prompt to Mistral LLM via Ollama
    response = ollama.chat(
        model='mistral',
        messages=[{'role': 'user', 'content': prompt}]
    )

    raw_output = response['message']['content']

    # Parse triples from LLM output using regex
    triples = re.findall(r'\((.*?)\)', raw_output)

    for triple in triples:
        parts = [p.strip() for p in triple.split(',')]

        if len(parts) == 3:
            subject, relation, obj = parts

            # Clean any accidental labels in the output
            subject  = subject.replace('Product Name:', '').strip()
            relation = relation.replace('RELATION:', '').strip()
            obj      = obj.replace('SPECIFIC DETAIL:', '').strip()

            # Filter out weak or generic outputs
            # These add no value to the knowledge graph
            if obj.lower() in ['unknown', 'unspecified', 'issue', 'none']:
                continue

            llm_triples.append([subject, relation, obj])

    print(f'  ✅ Ticket {index+1} processed — Product: {product}')

print(f'\n✅ LLM Triple Extraction Complete!')
print(f'   Total LLM Triples Extracted: {len(llm_triples)}')

---
## Step 8 — Structure and Save LLM Triples

### Objective
Convert the raw LLM triple list into a structured pandas DataFrame and save it as a CSV file.



In [ ]:
# Convert LLM triples list to a structured DataFrame
# Columns: Subject (entity), Predicate (relation), Object (entity)
llm_triples_df = pd.DataFrame(
    llm_triples,
    columns=['Subject', 'Predicate', 'Object']
)

# Save to CSV as a checkpoint
llm_triples_df.to_csv('llm_triples.csv', index=False)

print('✅ LLM Triples Saved Successfully!')
print(f'   Total Rows      : {len(llm_triples_df)}')
print(f'   Relation Types  : {llm_triples_df["Predicate"].unique().tolist()}')
print(f'   Unique Products : {llm_triples_df["Subject"].nunique()}')
print()
llm_triples_df.head(10)

---
## Step 9 — Merge Structured + LLM Triples into Final Dataset

### Objective
Combine the rule-based structured triples and the LLM-extracted semantic triples into one unified final dataset.



In [ ]:
# Load both triple files
structured_df = pd.read_csv('structured_triples.csv')
llm_df        = pd.read_csv('llm_triples.csv')

# Merge both DataFrames into one final dataset
# ignore_index=True resets the row index after merging
final_triples_df = pd.concat(
    [structured_df, llm_df],
    ignore_index=True
)

# Remove exact duplicate triples
final_triples_df = final_triples_df.drop_duplicates()

# Remove rows where any column is null
final_triples_df = final_triples_df.dropna(
    subset=['Subject', 'Predicate', 'Object']
)

# Save the final merged dataset
final_triples_df.to_csv('final_triples.csv', index=False)

print('✅ Final Triples Merged and Saved!')
print(f'   Structured Triples : {len(structured_df):,}')
print(f'   LLM Triples        : {len(llm_df):,}')
print(f'   Total Combined     : {len(final_triples_df):,}')
print(f'   Unique Predicates  : {final_triples_df["Predicate"].unique().tolist()}')
print()
final_triples_df.head(10)

---
## Step 10 — Build NetworkX Knowledge Graph

### Objective
Construct an in-memory directed knowledge graph using NetworkX from the final triples dataset.



In [ ]:
# Build in-memory Knowledge Graph using NetworkX
# DiGraph = Directed Graph (edges have direction)
G = nx.DiGraph()

# Add all triples as nodes and edges
for _, row in final_triples_df.iterrows():

    subject   = str(row['Subject']).strip()
    predicate = str(row['Predicate']).strip()
    obj       = str(row['Object']).strip()

    # Add subject and object as nodes in the graph
    G.add_node(subject)
    G.add_node(obj)

    # Add directed edge from subject to object with relation label
    # subject --[predicate]--> object
    G.add_edge(subject, obj, label=predicate)

print('✅ NetworkX Knowledge Graph Built Successfully!')
print('=' * 50)
print('GRAPH STATISTICS')
print('=' * 50)
print(f'   Total Nodes (Entities)     : {G.number_of_nodes():,}')
print(f'   Total Edges (Relationships): {G.number_of_edges():,}')
print(f'   Graph Type                 : Directed Graph')
print()

# Show sample nodes
sample_nodes = list(G.nodes())[:10]
print('Sample Nodes (Entities):')
for node in sample_nodes:
    print(f'  → {node}')

---
## Step 11 — Connect to Neo4j Graph Database

### Objective
Establish a connection to Neo4j Desktop running locally and verify it is working.

### What is Neo4j?
Neo4j is the world's leading graph database. It stores data as nodes and relationships — exactly matching the knowledge graph structure. It uses **Cypher Query Language** (similar to SQL but for graphs).



In [ ]:
from neo4j import GraphDatabase

URI      = 'bolt://localhost:7687'    
USERNAME = 'neo4j'                    
PASSWORD = '12345'       


driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session() as session:
    result = session.run("RETURN 'Connection Successful!' AS message")
    print(result.single()['message'])

driver.close()
print('✅ Neo4j Desktop connection verified!')
print(f'   URI      : {URI}')
print(f'   Username : {USERNAME}')

---
## Step 12 — Push Knowledge Graph to Neo4j

### Objective
Insert all final triples into Neo4j as nodes and relationships, creating the persistent enterprise knowledge graph.



In [ ]:
from neo4j import GraphDatabase
import re

URI      = 'bolt://localhost:7687'
USERNAME = 'neo4j'
PASSWORD = '12345'   

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

triples_df = pd.read_csv('final_triples.csv')
triples_df = triples_df.dropna(subset=['Subject', 'Predicate', 'Object'])


def clean_relationship(rel):
    rel = str(rel).upper()              
    rel = rel.replace(' ', '_')           
    rel = re.sub(r'[^A-Z0-9_]', '', rel)  
    return rel


def create_graph(tx, subject, predicate, obj):
    """
    Creates two entity nodes and one relationship between them.
    Uses MERGE to prevent duplicate nodes and relationships.
    
    tx       = Neo4j transaction object
    subject  = Source entity (e.g. LG Smart TV)
    predicate = Relationship type (e.g. HAS_ISSUE)
    obj      = Target entity (e.g. Overheating)
    """
    # Clean the relationship name for Neo4j compatibility
    relationship = clean_relationship(predicate)

    # Cypher query:
    # MERGE (a:Entity) — create node a if not exists
    # MERGE (b:Entity) — create node b if not exists
    # MERGE (a)-[r:RELATION]->(b) — create relationship if not exists
    query = f"""
    MERGE (a:Entity {{name: $subject}})
    MERGE (b:Entity {{name: $object}})
    MERGE (a)-[r:{relationship}]->(b)
    """
    tx.run(query, subject=str(subject), object=str(obj))

# ── Execute Graph Insertion ────────────────────────
print('⏳ Pushing knowledge graph to Neo4j...')
print('-' * 50)

with driver.session() as session:
    for i, (_, row) in enumerate(triples_df.iterrows()):

        # execute_write handles transactions with auto-retry
        session.execute_write(
            create_graph,
            row['Subject'],
            row['Predicate'],
            row['Object']
        )

        # Print progress every 100 rows
        if (i + 1) % 100 == 0:
            print(f'  ✅ {i+1:,} triples pushed to Neo4j...')

driver.close()

print()
print('✅ Graph Construction Completed Successfully!')
print(f'   Total Triples Pushed to Neo4j: {len(triples_df):,}')

---
## Step 13 — Graph Validation and Statistics

### Objective
Verify the knowledge graph was stored correctly in Neo4j by querying node and relationship counts, and running sample queries.





In [ ]:
# Verify the knowledge graph in Neo4j
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session() as session:

    
    node_count = session.run(
        'MATCH (n) RETURN count(n) as count'
    ).single()['count']

    rel_count = session.run(
        'MATCH ()-[r]->() RETURN count(r) as count'
    ).single()['count']

    lg_results = session.run("""
        MATCH (a:Entity)-[r]->(b:Entity)
        WHERE a.name CONTAINS 'LG'
        RETURN a.name AS source, type(r) AS relation, b.name AS target
        LIMIT 5
    """)


    critical_results = session.run("""
        MATCH (a:Entity)-[:HAS_PRIORITY]->(b:Entity)
        WHERE b.name = 'critical'
        RETURN a.name AS ticket, b.name AS priority
        LIMIT 5
    """)

    
    print('=' * 50)
    print('NEO4J GRAPH STATISTICS')
    print('=' * 50)
    print(f'   Total Nodes (Entities)     : {node_count:,}')
    print(f'   Total Relationships        : {rel_count:,}')
    print()

    print('Sample Query 1 — LG Product Relationships:')
    print('-' * 50)
    for record in lg_results:
        print(f'  {record["source"]} --[{record["relation"]}]--> {record["target"]}')

    print()
    print('Sample Query 2 — Critical Priority Tickets:')
    print('-' * 50)
    for record in critical_results:
        print(f'  {record["ticket"]} has priority: {record["priority"]}')

driver.close()
print()
print('✅ Graph validation complete!')

---
## ✅ Milestone 2 — Complete Summary

### Tasks Completed

| Step | Task | Status | Output |
|---|---|---|---|
| 1 | Environment Setup | ✅ Done | Libraries imported |
| 2 | Dataset Loading | ✅ Done | 8,469 tickets loaded |
| 3 | LLM Connection Test | ✅ Done | Mistral verified |
| 4 | Sample Extraction Test | ✅ Done | Format verified |
| 5 | Regex Parsing | ✅ Done | Triples parsed |
| 6 | Structured Triples | ✅ Done | `structured_triples.csv` |
| 7 | LLM-Based NER | ✅ Done | `llm_triples.csv` |
| 8 | Save LLM Triples | ✅ Done | CSV saved |
| 9 | Merge All Triples | ✅ Done | `final_triples.csv` |
| 10 | NetworkX Graph | ✅ Done | In-memory graph built |
| 11 | Neo4j Connection | ✅ Done | Connection verified |
| 12 | Push to Neo4j | ✅ Done | Graph stored in DB |
| 13 | Graph Validation | ✅ Done | Statistics confirmed |


### Files Generated
```
structured_triples.csv  → Rule-based triples from ticket columns
llm_triples.csv         → Semantic triples from Mistral LLM
final_triples.csv       → Merged complete knowledge graph dataset
Neo4j Database          → Persistent queryable graph storage
```

### Next Step → Milestone 3
The `final_triples.csv` and Neo4j graph will be used in **Milestone 3** to build the Semantic Search and RAG pipeline using FAISS vector embeddings and Mistral LLM for intelligent Q&A.